# Train the misconception-tagged AP Bio item generator (QLoRA)

Fine-tunes **Qwen/Qwen3-1.7B** on the by-construction generation dataset to instill the Behavior Spec
(`docs/behavior_spec.md`). Runtime: **Runtime > Change runtime type > T4/A100 GPU**.

Pipeline: install → get code+data → train (`scripts/train_gen_sft.py`) → sanity-generate →
(optional) push adapter to HF Hub → base-vs-tuned eval.

**Before running:** commit + push the repo (including `data/gen_sft_train.jsonl`,
`data/gen_sft_val.jsonl`, `data/eval_scenarios.jsonl`, `data/apbio_misconceptions.json`) so the clone
cell can fetch them — or use the upload fallback cell.

In [1]:
!nvidia-smi -L  # confirm a GPU is attached

GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-6ba91594-3fab-f8a6-aba3-2d4e210408ee)


In [2]:
# Unsloth QLoRA stack (~2x faster, ~70% less VRAM; fits a free T4).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 166.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 145.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 62.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 169.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 32.6 MB/s eta 0:00:00


In [3]:
# Get code + data. Set REPO to your fork's URL; the repo must already contain
# the data/*.jsonl files (commit them first).
REPO = "https://github.com/gabriel-xiong/algebra_error_classifier.git"
import os
if not os.path.isdir("algebra_error_classifier"):
    !git clone $REPO
%cd algebra_error_classifier
!ls data/gen_sft_train.jsonl data/gen_sft_val.jsonl data/eval_scenarios.jsonl

Cloning into 'algebra_error_classifier'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 119 (delta 32), reused 109 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 1.63 MiB | 4.51 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/algebra_error_classifier
data/eval_scenarios.jsonl  data/gen_sft_train.jsonl  data/gen_sft_val.jsonl


### Upload fallback (only if the clone lacks the data files)
Run the next cell to upload `gen_sft_train.jsonl`, `gen_sft_val.jsonl`, `eval_scenarios.jsonl`,
and `apbio_misconceptions.json` from your machine into `data/`.

In [ ]:
# from google.colab import files
# import shutil, os
# os.makedirs('data', exist_ok=True)
# up = files.upload()
# for name in up:
#     shutil.move(name, f'data/{name}')

In [5]:
# Train. Completion-only masking is applied inside the script (loss on the JSON only).
!python scripts/train_gen_sft.py \
    --model Qwen/Qwen3-1.7B \
    --train data/gen_sft_train.jsonl \
    --val   data/gen_sft_val.jsonl \
    --out   outputs/gen_lora \
    --epochs 3 --batch-size 4 --grad-accum 4 --max-seq-length 1536

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_lin

In [6]:
# Sanity check: load base+adapter and generate one item from a held-out scenario.
import json, sys
sys.path.insert(0, 'scripts')
from unsloth import FastLanguageModel
import gen_spec

model, tok = FastLanguageModel.from_pretrained('outputs/gen_lora', max_seq_length=1536, load_in_4bit=True)
FastLanguageModel.for_inference(model)

sc = json.loads(open('data/eval_scenarios.jsonl').readline())
system, user = gen_spec.build_generation_prompt(sc['topic'], sc['misconception_ids'])
msgs = [{'role':'system','content':system},{'role':'user','content':user}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = tok(text, return_tensors='pt').to(model.device)
out = model.generate(**ids, max_new_tokens=512, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking

<think>

</think>

{"stem":"What is true of glycolysis in a typical cell?","choices":{"A":"It feeds the electron transport chain, which runs equally well with or without oxygen.","B":"It is essentially fermentation and directly yields the bulk of the ATP.","C":"It takes place in the cytoplasm and can proceed whether or not oxygen is present.","D":"The NADH it makes is spent straight as ATP."},"correct":"C","distractor_tags":{"A":{"misconception_id":"cr_etc_runs_anaerobically"},"B":{"misconception_id":"cr_fermentation_makes_atp"},"D":{"misconception_id":"cr_nadh_makes_atp_directly"}}}


In [7]:
# Save the trained artifacts ("endpoints") to Google Drive so they survive the
# Colab session (local disk is wiped on disconnect). Run after training.
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/apbio_item_generator'
os.makedirs(DEST, exist_ok=True)

# Copy whatever exists: LoRA adapter, merged model, eval results.
for src in ['outputs/gen_lora', 'outputs/gen_merged', 'data/eval_results.jsonl']:
    if os.path.exists(src):
        dst = os.path.join(DEST, os.path.basename(src))
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print('saved ->', dst)
    else:
        print('skip (not found yet):', src)

print('\nDrive folder:', DEST)
!ls -la "$DEST"

Mounted at /content/drive
saved -> /content/drive/MyDrive/apbio_item_generator/gen_lora
skip (not found yet): outputs/gen_merged
saved -> /content/drive/MyDrive/apbio_item_generator/eval_results.jsonl

Drive folder: /content/drive/MyDrive/apbio_item_generator
total 19
-rw------- 1 root root 14883 Jul 10 16:20 eval_results.jsonl
drwx------ 3 root root  4096 Jul 10 16:25 gen_lora


In [13]:
from huggingface_hub import login
login()   # paste the hf_... token into the prompt
model.push_to_hub('gabriel-xiong/apbio-item-generator-qwen3-1.7b-lora')
tok.push_to_hub('gabriel-xiong/apbio-item-generator-qwen3-1.7b-lora')


README.md:   0%|          | 0.00/565 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 69.8MB            

Saved model to https://huggingface.co/gabriel-xiong/apbio-item-generator-qwen3-1.7b-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp0puxmia7/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp0puxmia7/tokenizer.json:  99%|#########8| 11.3MB / 11.4MB            

### Base-vs-tuned eval (the results table)
Genetics is scored objectively offline (no key). The **conceptual** dimensions need the calibrated
judge — set `OPENAI_API_KEY` in the cell below and pass `--judge-model gpt-4o` (gpt-4o-mini FAILED
calibration). Merge the LoRA into a full model first so `eval_generation.py`'s HF loader can read it.

In [8]:
# In-distribution base-vs-tuned eval.
# Merge adapter -> standalone dir that eval_generation.py loads with hf:
model.save_pretrained_merged('outputs/gen_merged', tok, save_method='merged_16bit')

import os
os.environ['OPENAI_API_KEY'] = "sk-...PASTE_KEY_HERE..."   # <-- paste between the quotes
assert os.environ['OPENAI_API_KEY'].startswith('sk-'), 'paste your key in the line above'

!python scripts/eval_generation.py \
    --base  hf:Qwen/Qwen3-1.7B \
    --tuned hf:outputs/gen_merged \
    --judge-model gpt-4o \
    --out data/eval_results.jsonl
# Header must read (gpt-4o), NOT MOCK.


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in outputs/gen_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:03<00:00,  3.56s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:12<00:00, 12.10s/it]


Unsloth: Merge process complete. Saved to `/content/algebra_error_classifier/outputs/gen_merged`
config.json: 100% 726/726 [00:00<00:00, 7.27MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 53.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 73.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 212MB/s]
tokenizer.json: 100% 11.4M/11.4M [00:00<00:00, 22.8MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors.index.json: 100% 25.6k/25.6k [00:00<00:00, 144MB/s]
Fetching 2 files: 100% 2/2 [00:03<00:00,  1.89s/it]
Download complete: 100% 4.06G/4.06G [00:03<00:00, 1.07GB/s]                
Loading weights: 10

In [9]:
# OUT-OF-DISTRIBUTION eval: photosynthesis (a topic the model NEVER trained on).
# Tests whether the BEHAVIOR generalizes vs. overfits to trained topics.
# Uses the same key from the cell above (already in os.environ) + temperature
# sampling so each scenario gives distinct generations.
!python scripts/eval_generation.py \
    --base  hf:Qwen/Qwen3-1.7B \
    --tuned hf:outputs/gen_merged \
    --scenarios data/eval_scenarios_ood.jsonl \
    --temperature 0.7 \
    --judge-model gpt-4o \
    --out data/eval_results_ood.jsonl
# Compare tuned distractor_mapping here (5-topic v2) vs the v1 value of 0.963.

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 311/311 [00:00<00:00, 1066.86it/s]
The tokenizer you are loading from 'outputs/gen_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Loading weights: 100% 310/310 [00:00<00:00, 1259.26it/s]
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the doc

In [11]:
# frontier model 
!git pull -q
!python scripts/eval_generation.py \
    --base  openai:gpt-4o \
    --tuned hf:outputs/gen_merged \
    --judge-model gpt-4o \
    --out data/eval_frontier.jsonl


The tokenizer you are loading from 'outputs/gen_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 310/310 [00:00<00:00, 1257.24it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=40960) seem 

In [12]:
!git pull -q
!python scripts/eval_generation.py \
    --base  openai:gpt-4o \
    --tuned hf:outputs/gen_merged \
    --scenarios data/eval_scenarios_ood.jsonl \
    --temperature 0.7 \
    --judge-model gpt-4o \
    --out data/eval_frontier_ood.jsonl


The tokenizer you are loading from 'outputs/gen_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 310/310 [00:00<00:00, 1254.19it/s]
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/